In [2]:
pip install chromadb

Note: you may need to restart the kernel to use updated packages.


In [1]:
# hugging face embedding models
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from sklearn.metrics.pairwise import cosine_similarity

# need to install the model as well, dimensions are default 384
embeddings = HuggingFaceEmbeddings(
    model='sentence-transformers/all-MiniLM-L6-v2')

/Users/khushboogour/Documents/Langchain/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 47178.48it/s]


In [2]:
document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
    id=1,
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
    id=2,
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
    id=3,
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
    id=4,
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
    id=5,
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    ]

In [3]:
vector_store = Chroma(embedding_function=embeddings,
                      persist_directory='chroma_database', collection_name='sample_data')

In [47]:
# will return unique id asociated with each vector ['1', '2', '3', '4', '5']
vector_store.add_documents(documents)

['1', '2', '3', '4', '5']

In [4]:
vector_store.get( ids=['1', '2', '3', '4', '5'], include=['embeddings', 'metadatas', "documents"])

{'ids': [],
 'embeddings': array([], dtype=float64),
 'documents': [],
 'uris': None,
 'included': ['embeddings', 'metadatas', 'documents'],
 'data': None,
 'metadatas': []}

In [35]:
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    # k=2,
    # filter={"source": "news"} #only applies to meta dat fields
)

results

# query_vec = embeddings.embed_query(
#     "LangChain provides abstractions to make working with LLMs easy")
# texts = [doc.page_content for doc in documents]

# doc_vec = embeddings.embed_documents(texts)
# scores = cosine_similarity([query_vec], doc_vec)[0]

# scores

[Document(id='3', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='4', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(id='1', metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(id='5', metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again.")]

In [5]:
results = vector_store.similarity_search_with_score(
    "LangChain provides abstractions to make working with LLMs easy",
    # k=2,
    # filter={"source": "news"} #only applies to meta dat fields
)
results

[]

In [6]:
vector_store.update_document(document_id="1", document= Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast last morning.",
    metadata={"source": "tweet"},
    id=1,
))

In [43]:
vector_store.delete(ids= ["3"])

In [ ]:
# FAISS

In [8]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(documents, embedding=embeddings)
vector_store

In [11]:
vector_store.similarity_search_with_score("LangChain provides abstractions to make working with LLMs easy",k=1)

[(Document(id='3', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
  np.float32(0.8416443))]

In [16]:
# to store permanently in disk

vector_store.save_local('faiss,indexes')

new_db = vector_store.load_local('faiss,indexes', embeddings,allow_dangerous_deserialization=True)

In [17]:
new_db.similarity_search_with_score("LangChain provides abstractions to make working with LLMs easy", k=1)

[(Document(id='3', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
  np.float32(0.8416443))]